# ICA — apply

Applies a previously fitted decomposition. Never refits.


In [ ]:
subject_id = "170"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. derivatives_root -> a test folder
path_overrides = None


In [ ]:
import mne
from mne.preprocessing import ICA
import autoreject
from pathlib import Path
import pandas as pd
from datetime import datetime
import json
import os

from spectral.viz import plot_step, plot_bad_channels, plot_epochs
from spectral.specparam import specparam2pandas
from spectral.utils import ProjectPaths,print_timestamp
from spectral.ica import compute_ica_limited, plot_annotated_topographies, plot_component_properties,get_manual_ica_list
from mne_icalabel.gui import label_ica_components
from mne_icalabel import label_components


""" config = read_parameters()
my_paths = config_project(subject)
project_path = my_paths["project_path"]
figures_path = my_paths["figures_path"]
specparam_path = my_paths["specparam_path"]

Path(specparam_path).mkdir(parents=True, exist_ok=True)
 """


paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
# Create all directories
paths.create_directories()

#paths = ProjectPaths(subject)
# Create all directories
#paths.create_directories()

# This is analysis output, so it goes in the analysis folder
specparam_path = paths.specparam
project_path = paths.root
figures_path = paths.figures
preprocessed_path = paths.preprocessed
epochs_path = paths.epochs
# Print paths to verify
print_timestamp("Setting up project paths")
paths.show()

mne.viz.set_browser_backend("matplotlib")
# mne.viz.set_browser_backend("qt")
mne.set_config("MNE_BROWSER_THEME", "light")
# matplotlib.use("Agg")
#print_date_time()

In [ ]:
clean_raw = mne.io.read_raw_fif(
    f"{paths.preprocessed}/sub-{subject_id}_annotated_filtered_raw.fif", preload=True
)

total_duration = clean_raw.times[-1]
clean_raw = clean_raw.crop(tmin=3.0, tmax=total_duration - 3, include_tmax=True)

report = mne.Report(title=f"Report for subject {subject_id}", subject=subject_id)
report.add_raw(raw=clean_raw, title="Raw", psd=True, butterfly=True, scalings='auto')

# --- HANDLES CASE WITH NO BAD CHANNELS ---
if clean_raw.info["bads"]:
    print(f"Plotting {len(clean_raw.info['bads'])} bad channels for the report.")
    
    # We use a try-except here because plotting bads can occasionally 
    # throw a ValueError if channels were marked bad but then dropped.
    try:
        bad_channel_plot = clean_raw.plot(
            picks=clean_raw.info["bads"],
            duration=min(300.0, total_duration - 6), # Ensure duration isn't longer than file
            scalings=dict(eeg=1e-4),
            show_scrollbars=False,
            show=False # Prevent popup in headless environments
        )
        report.add_figure(bad_channel_plot, title="Rejected bad channels")
    except Exception as e:
        print(f"Could not plot bad channels: {e}")
else:
    print("No bad channels detected. Skipping bad channel plot.")
    # Optional: Add a text note to the report instead of a plot
    report.add_html(
        html="<p><b>Note:</b> No bad channels were detected or marked for this subject.</p>",
        title="Rejected bad channels"
    )

In [ ]:
epochs_good = mne.read_epochs(
    f"{paths.epochs}/sub-{subject_id}_good_epochs-epo.fif"
)

In [ ]:
from spectral.ica import (
    apply_ica_decision, write_ica_application, describe_ica_application,
)
from html import escape
import json as _json

ica = mne.preprocessing.read_ica(
    paths.analysis / f"sub-{subject_id}_my_ica_model-ica.fif")
decision_path = paths.analysis / f"sub-{subject_id}_ica-decision.json"
decision = _json.loads(decision_path.read_text())
application_path = paths.analysis / f"sub-{subject_id}_ica-application.json"


In [ ]:
epochs_clean, application = apply_ica_decision(
    epochs_good, ica, decision, corr_threshold=0.95,
    input_source=paths.epochs / f"sub-{subject_id}_good_epochs-epo.fif")
saved_epoch_files = epochs_clean.save(
    paths.analysis / f"sub-{subject_id}_clean_ica-epo.fif", overwrite=True)
application = write_ica_application(
    application_path, application, epochs_files=saved_epoch_files)
summary = describe_ica_application(application)
print(summary)
report.add_html(f"<p>{escape(summary)}</p>", title="ICA application", replace=True)


In [ ]:
cleaned_plot = ica.plot_overlay(epochs_good.average())

In [ ]:
epochs_plot, epochs_timeseries, epochs_plot_psd = plot_epochs(
epochs_clean, figures_path=figures_path, subject=subject_id
)
report.add_figure(epochs_plot, title="Cleaned epochs after ICA")
report.add_figure(epochs_plot_psd, title="Cleaned epochs after ICA - PSD")

In [ ]:
report.save(
    f"{paths.reports}/sub-{subject_id}_report_04_ica_apply.html", overwrite=True
)